In [ ]:
from flask import Flask, request, jsonify
import pandas as pd
import sqlite3
from datetime import datetime
import os

app = Flask(__name__)

# ==============================
# DATABASE SETUP
# ==============================

DATABASE = "advertiser_data.db"

conn = sqlite3.connect(DATABASE)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS advertiser_data (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    advertiser_name TEXT,
    campaign_name TEXT,
    impressions INTEGER,
    clicks INTEGER,
    country TEXT,
    platform TEXT,
    budget INTEGER,
    ctr REAL,
    upload_time TEXT
)
""")

conn.commit()
conn.close()

# ==============================
# CREATE UPLOAD FOLDER
# ==============================

UPLOAD_FOLDER = "uploads"

if not os.path.exists(UPLOAD_FOLDER):
    os.makedirs(UPLOAD_FOLDER)

# ==============================
# CTR CALCULATION
# ==============================

def calculate_ctr(clicks, impressions):

    if impressions == 0:
        return 0

    return round((clicks / impressions) * 100, 2)

# ==============================
# DATA VALIDATION
# ==============================

def validate_data(df):

    required_columns = [
        'advertiser_name',
        'campaign_name',
        'impressions',
        'clicks',
        'country',
        'platform',
        'budget'
    ]

    missing_columns = []

    for col in required_columns:
        if col not in df.columns:
            missing_columns.append(col)

    return missing_columns

# ==============================
# HOME ROUTE
# ==============================

@app.route('/')
def home():
    return "Advertiser Data Upload API Running Successfully"

# ==============================
# FILE UPLOAD API
# ==============================

@app.route('/upload', methods=['POST'])
def upload_file():

    if 'file' not in request.files:
        return jsonify({
            "status": "FAILED",
            "message": "No file uploaded"
        })

    file = request.files['file']

    filepath = os.path.join(UPLOAD_FOLDER, file.filename)

    file.save(filepath)

    try:

        # READ EXCEL FILE
        df = pd.read_excel(filepath)

        # VALIDATE DATA
        missing_columns = validate_data(df)

        if missing_columns:
            return jsonify({
                "status": "FAILED",
                "missing_columns": missing_columns
            })

        # DATABASE CONNECTION
        conn = sqlite3.connect(DATABASE)
        cursor = conn.cursor()

        records_uploaded = 0

        for _, row in df.iterrows():

            ctr = calculate_ctr(
                int(row['clicks']),
                int(row['impressions'])
            )

            cursor.execute("""
            INSERT INTO advertiser_data (
                advertiser_name,
                campaign_name,
                impressions,
                clicks,
                country,
                platform,
                budget,
                ctr,
                upload_time
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                row['advertiser_name'],
                row['campaign_name'],
                int(row['impressions']),
                int(row['clicks']),
                row['country'],
                row['platform'],
                int(row['budget']),
                ctr,
                datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            ))

            records_uploaded += 1

        conn.commit()
        conn.close()

        return jsonify({
            "status": "SUCCESS",
            "message": "Data uploaded successfully",
            "records_uploaded": records_uploaded
        })

    except Exception as e:

        return jsonify({
            "status": "ERROR",
            "message": str(e)
        })

# ==============================
# FETCH DATA API
# ==============================

@app.route('/advertisers', methods=['GET'])
def get_data():

    conn = sqlite3.connect(DATABASE)

    query = "SELECT * FROM advertiser_data"

    df = pd.read_sql_query(query, conn)

    conn.close()

    return jsonify(df.to_dict(orient='records'))

# ==============================
# MAIN FUNCTION
# ==============================

if __name__ == '__main__':

    app.run(
        host='0.0.0.0',
        port=5000,
        debug=False
    )